# AeroFleet — 1,000-case mass forensics run on Colab (free GPU)

Colab-side counterpart to `research/kaggle_mass_forensics_run.ipynb`. Same harness, same real
Ollama backend, same model substitution — this notebook runs a **different, non-overlapping
batch range** so it can work in parallel with the Kaggle run (and a third machine, if you have
one) instead of redoing the same cases.

## Required one-time manual setup (Colab UI, not this notebook)

1. **Runtime → Change runtime type → T4 GPU**, then Save (this reconnects the runtime).
2. **Secrets** (key icon in the left sidebar) → **Add new secret** → name it `GH_PAT`, value =
   a GitHub fine-grained token, read-only, scoped to just `AdityaPathare46/aerofleet` (private
   repo). Toggle **Notebook access** on for this notebook.
3. This notebook mounts your Google Drive and writes results there directly (`/content/drive/...`)
   instead of relying on a Kaggle-style "Save Version" — that means progress survives a
   disconnect automatically, no manual save step needed at session end.

## Which batch range this notebook runs

`--target 1000 --batch-size 25` gives 40 batches total. Suggested 3-way split across
Kaggle / Colab / a third machine:

| Machine | Batches | Cases |
|---|---|---|
| Kaggle | 1-14 | MFI-00001 – MFI-00350 |
| **This Colab notebook** | **27-40** | **MFI-00651 – MFI-01000** |
| Third machine (friend) | 15-26 | MFI-00351 – MFI-00650 |

Change `ONLY_BATCHES` in the run cell below if you want a different split — just make sure all
three machines' ranges are disjoint and together cover 1-40, or you'll either duplicate work or
leave a gap.

## Same model-substitution caveat as the Kaggle run

`llama4:scout` (67GB) doesn't fit a free GPU. The 4 agents it powers (DISPATCHER,
AIRSPACE_SAFETY, AI_VALIDATOR, CONTINGENCY) use `mistral-nemo:12b` instead here too — **this must
match exactly what the Kaggle and third-machine runs use**, or the pooled results mix two
different "real" configurations into one dataset without that being traceable per-case.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 1. Mount Drive (for persistence) and clone the private repo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from google.colab import userdata
_token = userdata.get('GH_PAT')

REPO_DIR = "/content/aerofleet"
import os
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://{_token}@github.com/AdityaPathare46/aerofleet.git {REPO_DIR}
else:
    print("Repo already present, skipping clone.")

del _token
%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt


## 2. Install Ollama and start the server in this session


In [ ]:
# zstd: required by the Ollama installer to extract its archive.
# pciutils (lspci): lets the installer auto-detect the GPU and install the
# matching CUDA runtime — without it, install silently warns and may fall
# back to a CPU-only build, which would make the run far too slow to finish.
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time, requests

log = open("/content/ollama_serve.log", "a")
proc = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=5)
        print("Ollama server is up.")
        break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not come up — check /content/ollama_serve.log")

time.sleep(2)
!grep -i -E "gpu|cuda|library=" /content/ollama_serve.log | tail -5


## 3. Pull the models

Same 4 models as the Kaggle run (~41GB total). Re-run this cell if a pull is interrupted —
Ollama resumes partial downloads.


In [ ]:
for model in ["gemma4:12b", "phi4-reasoning:plus", "mistral-small3.2", "mistral-nemo:12b"]:
    print(f"--- pulling {model} ---")
    !ollama pull {model}

!ollama list


## 4. Study directory on Drive (persists automatically) + this machine's batch range


In [ ]:
from pathlib import Path

STUDY_DIR = Path("/content/drive/MyDrive/aerofleet_mass_forensics_colab")
STUDY_DIR.mkdir(parents=True, exist_ok=True)

ONLY_BATCHES = "27-40"  # this machine's assigned, non-overlapping slice — see the table above


## 5. Configure the model substitution and record it (must match the other machines)


In [ ]:
import os, json, subprocess, datetime

os.environ["OLLAMA_HOST"] = "http://localhost:11434"
os.environ.pop("USE_MOCK_AGENTS", None)

for agent_id in ["DISPATCHER", "AIRSPACE_SAFETY", "AI_VALIDATOR", "CONTINGENCY"]:
    os.environ[f"AGENT_MODEL_{agent_id}"] = "mistral-nemo:12b"

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()

metadata = {
    "platform": "colab",
    "gpu": gpu_name,
    "only_batches": ONLY_BATCHES,
    "run_started_utc": datetime.datetime.utcnow().isoformat(),
    "substitution": {
        "replaced_model": "llama4:scout",
        "substitute_model": "mistral-nemo:12b",
        "reason": "llama4:scout is 67GB (109B-param MoE); does not fit a free-tier GPU",
        "affected_agents": ["DISPATCHER", "AIRSPACE_SAFETY", "AI_VALIDATOR", "CONTINGENCY"],
    },
}
with open(STUDY_DIR / "cloud_run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


## 6. Run the harness for this session, scoped to this machine's batch range

Writing straight to Drive means a disconnect loses at most the one in-flight case — no
end-of-session save step needed. Just reopen this notebook later and re-run from here; it
resumes automatically.


In [ ]:
!python -m scenario_engine.mass_forensics_evaluation \
    --study-dir {STUDY_DIR} \
    --target 1000 \
    --batch-size 25 \
    --only-batches {ONLY_BATCHES}


## 7. Check progress at any point


In [ ]:
results_file = STUDY_DIR / "results.jsonl"
if results_file.exists():
    with open(results_file) as f:
        n = sum(1 for _ in f)
    print(f"{n} case-attempts recorded so far in this machine's range ({ONLY_BATCHES}).")
else:
    print("No results yet.")


## 8. Merging all three machines' results

Once Kaggle, this Colab notebook, and the third machine have each made progress (they don't all
need to be *finished* — merging is safe at any point), pull all three `results.jsonl` +
`dataset_manifest.json` + `run_config.json` sets down to one place and run:

```bash
python -m research.merge_batched_results \
    --source /path/to/kaggle_study \
    --source /path/to/colab_study \
    --source /path/to/friend_study \
    --output /path/to/merged_study

python -m scenario_engine.mass_forensics_evaluation --study-dir /path/to/merged_study --report-only
```

The merge script refuses to combine studies that weren't generated with the same
`--target`/`--batch-size`/`--seed` (they wouldn't share a manifest), and warns if the same
case_id shows up from two sources — a sign the batch ranges weren't actually disjoint.
